# Notebook 04 — Matrix Factorization con SGD

**Objetivo:** Implementar y evaluar Collaborative Filtering mediante Matrix Factorization con Stochastic Gradient Descent.

## Fundamento Matemático

**Objetivo:** Factorizar $R \approx P \cdot Q^T$ donde:
- $R \in \mathbb{R}^{n_u \times n_i}$: matriz de ratings (0 = desconocido)
- $P \in \mathbb{R}^{n_u \times k}$: factores latentes de usuarios
- $Q \in \mathbb{R}^{n_i \times k}$: factores latentes de películas

**Función de pérdida con regularización L2:**
$$\mathcal{L} = \sum_{(u,i) \in \Omega} \left(r_{ui} - p_u \cdot q_i\right)^2 + \lambda\left(\|p_u\|^2 + \|q_i\|^2\right)$$

**Gradientes:**
$$\frac{\partial \mathcal{L}}{\partial p_u} = -2 \cdot e_{ui} \cdot q_i + 2\lambda p_u \qquad \frac{\partial \mathcal{L}}{\partial q_i} = -2 \cdot e_{ui} \cdot p_u + 2\lambda q_i$$

**Reglas de actualización SGD** (derivadas negativas escaladas por $\alpha$):
$$p_u \leftarrow p_u + \alpha \left(e_{ui} \cdot q_i - \lambda \cdot p_u\right)$$
$$q_i \leftarrow q_i + \alpha \left(e_{ui} \cdot p_u - \lambda \cdot q_i\right)$$

donde $e_{ui} = r_{ui} - p_u \cdot q_i$ es el error de predicción.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys, time

sys.path.insert(0, os.path.abspath('..'))

from src.models.matrix_factorization import MatrixFactorizationSGD
from src.evaluation.metrics import rmse, mae

## 1. Cargar Datos

In [ ]:
train_matrix = np.load('../data/processed/ratings_train.npy')
test_matrix  = np.load('../data/processed/ratings_test.npy')
movies_cf    = pd.read_csv('../data/processed/movies_cf.csv')

print(f'Train matrix: {train_matrix.shape}')
print(f'Test matrix:  {test_matrix.shape}')
print(f'Ratings conocidos en train: {(train_matrix > 0).sum()}')
print(f'Ratings conocidos en test:  {(test_matrix > 0).sum()}')

## 2. Entrenamiento con Hiperparámetros Base

In [ ]:
# Hiperparámetros iniciales
model = MatrixFactorizationSGD(
    k       = 20,       # dimensión del espacio latente
    alpha   = 0.005,    # tasa de aprendizaje
    lambda_ = 0.02,     # regularización L2
    epochs  = 100,
    seed    = 42
)

print('Entrenando modelo base...')
t0 = time.time()
model.fit(train_matrix, test_matrix=test_matrix, verbose=True)
print(f'\nTiempo de entrenamiento: {time.time() - t0:.1f}s')

## 3. Curva de Convergencia

**Esta gráfica es clave para el informe:** muestra que SGD efectivamente minimiza la pérdida y que el modelo no está sobreajustando.

In [ ]:
epochs_range = range(1, len(model.train_rmse_history) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, model.train_rmse_history, label='Train RMSE', color='steelblue', linewidth=2)
ax.plot(epochs_range, model.test_rmse_history,  label='Test RMSE',  color='coral',     linewidth=2, linestyle='--')

ax.set_xlabel('Época')
ax.set_ylabel('RMSE')
ax.set_title('Curva de Convergencia — Matrix Factorization con SGD')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/sgd_convergence.png', dpi=150)
plt.show()

print(f'\nTrain RMSE final: {model.train_rmse_history[-1]:.4f}')
print(f'Test  RMSE final: {model.test_rmse_history[-1]:.4f}')

## 4. Grid Search de Hiperparámetros

In [ ]:
# Explorar diferentes valores de k (dimensión latente) y alpha
k_values     = [5, 10, 20, 50]
alpha_values = [0.001, 0.005, 0.01]

results = []

for k in k_values:
    for alpha in alpha_values:
        m = MatrixFactorizationSGD(k=k, alpha=alpha, lambda_=0.02, epochs=50, seed=42)
        m.fit(train_matrix, verbose=False)

        # Evaluar en test
        t_users, t_movies = np.where(test_matrix > 0)
        preds  = np.sum(m.P[t_users] * m.Q[t_movies], axis=1)
        actual = test_matrix[t_users, t_movies]
        test_rmse_val = rmse(actual, preds)

        results.append({'k': k, 'alpha': alpha, 'test_rmse': test_rmse_val})
        print(f'k={k:2d}, alpha={alpha:.3f} → Test RMSE: {test_rmse_val:.4f}')

results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_rmse'].idxmin()]
print(f'\nMejores hiperparámetros: k={best["k"]:.0f}, alpha={best["alpha"]:.3f} → RMSE={best["test_rmse"]:.4f}')

In [ ]:
# Visualizar: RMSE vs k para cada alpha
fig, ax = plt.subplots(figsize=(10, 5))

for alpha in alpha_values:
    subset = results_df[results_df['alpha'] == alpha]
    ax.plot(subset['k'], subset['test_rmse'], marker='o', label=f'alpha={alpha}')

ax.set_xlabel('k (dimensión latente)')
ax.set_ylabel('Test RMSE')
ax.set_title('Grid Search: RMSE vs k por diferentes learning rates')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/mf_grid_search.png', dpi=150)
plt.show()

## 5. Modelo Final con Mejores Hiperparámetros

In [ ]:
best_k     = int(best['k'])
best_alpha = float(best['alpha'])

final_model = MatrixFactorizationSGD(k=best_k, alpha=best_alpha, lambda_=0.02, epochs=100, seed=42)
final_model.fit(train_matrix, test_matrix=test_matrix, verbose=True)

# Calcular métricas finales sobre el conjunto de test
t_users, t_movies = np.where(test_matrix > 0)
test_preds  = np.sum(final_model.P[t_users] * final_model.Q[t_movies], axis=1)
test_actual = test_matrix[t_users, t_movies]

print(f'\n=== Métricas Finales (Test) ===')
print(f'RMSE: {rmse(test_actual, test_preds):.4f}')
print(f'MAE:  {mae(test_actual, test_preds):.4f}')

## 6. Recomendaciones de Ejemplo

In [ ]:
# Mostrar recomendaciones para 3 usuarios de ejemplo
for user_id in [0, 1, 2]:
    top10 = final_model.recommend(user_id, train_matrix, n=10)
    movie_titles = movies_cf.iloc[top10]['title'].tolist()
    print(f'\nRecomendaciones para Usuario {user_id}:')
    for rank, title in enumerate(movie_titles, 1):
        print(f'  {rank:2d}. {title}')

In [ ]:
# Guardar predicciones para el notebook de comparación
np.save('../data/processed/mf_P.npy', final_model.P)
np.save('../data/processed/mf_Q.npy', final_model.Q)
print('Factores latentes guardados.')